CONFIGURE THE ENVIRONMENT

In [ ]:
!conda install -y -c conda-forge openjdk

In [ ]:
!cd /tmp && wget https://archive.apache.org/dist/solr/solr/9.6.1/solr-9.6.1.tgz
!cd /tmp && tar -xzf solr-9.6.1.tgz

In [ ]:
!docker run -d -p 8983:8983 --name my-solr solr:latest

In [ ]:
!pip install pysolr

In [ ]:
!pip install pysolr requests

In [ ]:
!/tmp/solr-9.6.1/bin/solr start

In [ ]:
from pysolr import Solr
import time

time.sleep(3)
solr = Solr('http://localhost:8983/solr')
print("Conectado ao Solr!")

In [10]:
!find /tmp -name "solr" -type d

/tmp/solr-9.6.1/server/solr
/tmp/solr-9.6.1/server/solr-webapp/webapp/libs/solr


In [12]:
!/tmp/solr-9.6.1/bin/solr status


Found 1 Solr nodes: 

Solr process 1646270 running on port 8983
{
  "solr_home":"/tmp/solr-9.6.1/server/solr",
  "version":"9.6.1 d7f7166567f52f1b31e3315b0188e11f2c4c9b60 - houston - 2024-05-23 13:50:22",
  "startTime":"Tue Oct 14 21:01:22 UTC 2025",
  "uptime":"0 days, 1 hours, 20 minutes, 58 seconds",
  "memory":"287.8 MB (%56.2) of 512 MB"}



In [11]:
!/tmp/solr-9.6.1/bin/solr create -c municipios

         To turn off: bin/solr config -c municipios -p 8983 -action set-user-property -property update.autoCreateFields -value false

Created new core 'municipios'


In [13]:
!/tmp/solr-9.6.1/bin/solr create -c modalidades

         To turn off: bin/solr config -c modalidades -p 8983 -action set-user-property -property update.autoCreateFields -value false

Created new core 'modalidades'


LET'S TO THE PIPELINE

In [14]:
# Importar bibliotecas e configurações iniciais
import pysolr
import requests
import pandas as pd
import re
from typing import List, Dict
import json

# Configurações
SOLR_URL_BASE = "http://localhost:8983/solr"
UF_SC_ID = 42  # Código IBGE para Santa Catarina

print("Bibliotecas importadas com sucesso!")
print(f"URL do Solr: {SOLR_URL_BASE}")

Bibliotecas importadas com sucesso!
URL do Solr: http://localhost:8983/solr


In [19]:
#Baixar lista de municípios de SC via API IBGE
print("Baixando lista de municípios de Santa Catarina via API IBGE...")

url = f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{UF_SC_ID}/municipios"
municipios_data = requests.get(url).json()

# Criar DataFrame com os municípios
municipios_df = pd.DataFrame(municipios_data)
municipios_df = municipios_df[['nome']]  # Seleciona apenas essas colunas
municipios_df.rename(columns={'nome': 'municipio'}, inplace=True)

print(f"\nTotal de municípios encontrados: {len(municipios_df)}")
print("\nPrimeiros 10 municípios:")
print(municipios_df.head(10))

Baixando lista de municípios de Santa Catarina via API IBGE...

Total de municípios encontrados: 295

Primeiros 10 municípios:
          municipio
0     Abdon Batista
1      Abelardo Luz
2        Agrolândia
3        Agronômica
4         Água Doce
5  Águas de Chapecó
6       Águas Frias
7      Águas Mornas
8    Alfredo Wagner
9   Alto Bela Vista


In [15]:
# Criar lista de modalidades de licitação
# Lista das 13 modalidades de licitação no Brasil (conforme Lei 8.666/93 e Lei 14.133/21)
modalidades = [
    "Concorrência",
    "Concurso",
    "Pregão Eletrônico",
    "Pregão Presencial",
    "Tomada de Preços",
    "Convite",
    "Dispensa de Licitação",
    "Inexigibilidade de Licitação",
    "Leilão",
    "Procedimento Simplificado",
    "Regime diferenciado de contratação",
    "Procedimento Licitatório Lei 13.303/06"
]

# Criar DataFrame com as modalidades
modalidades_df = pd.DataFrame({
    'id': range(1, len(modalidades) + 1),
    'modalidade': modalidades
})

print(f"Total de modalidades de licitação: {len(modalidades_df)}")
print("\nModalidades cadastradas:")
for idx, row in modalidades_df.iterrows():
    print(f"  {row['id']:2d}. {row['modalidade']}")

Total de modalidades de licitação: 12

Modalidades cadastradas:
   1. Concorrência
   2. Concurso
   3. Pregão Eletrônico
   4. Pregão Presencial
   5. Tomada de Preços
   6. Convite
   7. Dispensa de Licitação
   8. Inexigibilidade de Licitação
   9. Leilão
  10. Procedimento Simplificado
  11. Regime diferenciado de contratação
  12. Procedimento Licitatório Lei 13.303/06


In [21]:
# Indexar municípios no Solr
print("Indexando municípios no Solr...\n")

solr_municipios = pysolr.Solr(f'{SOLR_URL_BASE}/municipios', always_commit=True, timeout=10)

# Preparar documentos para indexar
docs_municipios = []
for idx, row in municipios_df.iterrows():
    docs_municipios.append({
        'id': str(idx),
        'municipio': row['municipio'],
        'municipio_lower': row['municipio'].lower()
    })

docs_municipios = [
    {'id': str(idx), 'municipio': row['municipio'], 'municipio_lower': row['municipio'].lower()}
    for idx, (_, row) in enumerate(municipios_df.iterrows())
]

# Limpar o core antes de indexar (opcional, mas recomendado)
solr_municipios.delete(q='*:*')

# Indexar os documentos
solr_municipios.add(docs_municipios)

print(f"{len(docs_municipios)} municípios indexados com sucesso!")

# Verificação: buscar um teste
resultado_teste = solr_municipios.search('*:*', rows=1)
print(f"\nVerificação: {resultado_teste.hits} documentos no core 'municipios'")
print(f"   Exemplo: {docs_municipios[0]}")

Indexando municípios no Solr...

295 municípios indexados com sucesso!

Verificação: 295 documentos no core 'municipios'
   Exemplo: {'id': '0', 'municipio': 'Abdon Batista', 'municipio_lower': 'abdon batista'}


In [23]:
# Indexar modalidades no Solr
print("Indexando modalidades de licitação no Solr...\n")

solr_modalidades = pysolr.Solr(f'{SOLR_URL_BASE}/modalidades', always_commit=True, timeout=10)

# Preparar documentos para indexar
docs_modalidades = []
for idx, row in modalidades_df.iterrows():
    docs_modalidades.append({
        'id': str(row['id']),
        'modalidade': row['modalidade'],
        'modalidade_lower': row['modalidade'].lower()
    })

# Limpar e indexar
solr_modalidades.delete(q='*:*')
solr_modalidades.add(docs_modalidades)

print(f"{len(docs_modalidades)} modalidades indexadas com sucesso!")

# Verificação
resultado_teste = solr_modalidades.search('*:*', rows=1)
print(f"\nVerificação: {resultado_teste.hits} documentos no core 'modalidades'")
print(f"   Exemplo: {docs_modalidades[0]}")

Indexando modalidades de licitação no Solr...

12 modalidades indexadas com sucesso!

Verificação: 12 documentos no core 'modalidades'
   Exemplo: {'id': '1', 'modalidade': 'Concorrência', 'modalidade_lower': 'concorrência'}


In [35]:
def encontrar_municipios(texto, solr_conn):
    """
    Procura por municípios de SC mencionados no texto.
    
    Args:
        texto: string com o texto da notícia
        solr_conn: conexão com o core 'municipios' no Solr
    
    Returns:
        Lista de nomes de municípios encontrados
    """
    texto_lower = texto.lower()
    municipios_encontrados = []
    
    # Buscar todos os municípios indexados
    for doc in solr_conn.search('*:*', rows=300):
        municipio_lower = doc['municipio_lower']
        # Tratar caso onde Solr retorna lista em vez de string
        if isinstance(municipio_lower, list):
            municipio_lower = municipio_lower[0]
        
        # Buscar com regex (word boundary) para evitar correspondências parciais
        if re.search(rf"\b{re.escape(municipio_lower)}\b", texto_lower):
            municipio = doc['municipio']
            if isinstance(municipio, list):
                municipio = municipio[0]
            municipios_encontrados.append(municipio)
    
    return municipios_encontrados


def encontrar_modalidades(texto, solr_conn):
    """
    Procura por modalidades de licitação mencionadas no texto.
    
    Args:
        texto: string com o texto da notícia
        solr_conn: conexão com o core 'modalidades' no Solr
    
    Returns:
        Lista de modalidades encontradas
    """
    texto_lower = texto.lower()
    modalidades_encontradas = []
    
    # Buscar todas as modalidades indexadas
    for doc in solr_conn.search('*:*', rows=20):
        modalidade_lower = doc['modalidade_lower']
        # Tratar caso onde Solr retorna lista em vez de string
        if isinstance(modalidade_lower, list):
            modalidade_lower = modalidade_lower[0]
        
        # Buscar com regex
        if re.search(rf"\b{re.escape(modalidade_lower)}\b", texto_lower):
            modalidade = doc['modalidade']
            if isinstance(modalidade, list):
                modalidade = modalidade[0]
            modalidades_encontradas.append(modalidade)
    
    return modalidades_encontradas


print("Funções de extração criadas!")

Funções de extração criadas!


In [57]:
def encontrar_editais(texto: str) -> List[str]:
    """
    Extrai números de editais do texto.
    
    Padrões identificados:
    - 13/2021
    - CC184/2021
    - n. 36/2018
    - nº 45/2022
    
    Args:
        texto: string com o texto da notícia
    
    Returns:
        Lista de strings com os números de editais encontrados
    """
    editais_encontrados = []
    
    # Padrão: captura variações como "13/2021", "CC184/2021", "n. 36/2018", "nº 45/2022"
    # \b = word boundary (início/fim de palavra)
    # (?:n\.?|nº)?\s* = opcional: "n.", "n", "nº" seguido de espaços opcionais
    # [A-Z]{0,3} = 0 a 3 letras maiúsculas (ex: CC, TP)
    # \d+ = um ou mais dígitos
    # /20[12]\d = barra seguida de ano 201X ou 202X
    
    pattern = r'\b(?:n\.?|nº)?\s*([A-Z]{0,3}\d+/20[12]\d)\b'
    
    matches = re.finditer(pattern, texto, re.IGNORECASE)
    
    for match in matches:
        edital = match.group(1)  # Captura apenas o número (sem "n." ou "nº")
        if edital not in editais_encontrados:  # Evita duplicatas
            editais_encontrados.append(edital)
    
    return editais_encontrados


print("Função encontrar_editais() criada!")
print("\nTestando com exemplos:")

# Testes
exemplos = [
    "empresários manipularam o pregão presencial n. 36/2018",
    "na forma originalmente licitada no processo licitatório n. 013/2022",
    "O edital 13/2021 foi cancelado",
    "Concorrência CC184/2021 sob investigação"
]

for ex in exemplos:
    editais = encontrar_editais(ex)
    print(f"\nTexto: {ex}")
    print(f"Editais: {editais}")

Função encontrar_editais() criada!

Testando com exemplos:

Texto: empresários manipularam o pregão presencial n. 36/2018
Editais: ['36/2018']

Texto: na forma originalmente licitada no processo licitatório n. 013/2022
Editais: ['013/2022']

Texto: O edital 13/2021 foi cancelado
Editais: ['13/2021']

Texto: Concorrência CC184/2021 sob investigação
Editais: ['CC184/2021']


In [36]:
# Reconectar aos cores
solr_municipios = pysolr.Solr(f'{SOLR_URL_BASE}/municipios', timeout=10)
solr_modalidades = pysolr.Solr(f'{SOLR_URL_BASE}/modalidades', timeout=10)

# Exemplo 1 de notícia (do seu documento)
noticia_1 = """
A prefeitura de Chapecó, em Santa Catarina, abriu um pregão eletrônico para 
a construção de uma ponte na localidade Linha Salgado. A licitação foi 
cancelada após investigações descobrirem fraude no processo.
"""

print("NOTÍCIA 1:")
print(noticia_1)

municipios = encontrar_municipios(noticia_1, solr_municipios)
modalidades = encontrar_modalidades(noticia_1, solr_modalidades)

print("\n Municípios:", municipios)
print(" Modalidades:", modalidades)

NOTÍCIA 1:

A prefeitura de Chapecó, em Santa Catarina, abriu um pregão eletrônico para 
a construção de uma ponte na localidade Linha Salgado. A licitação foi 
cancelada após investigações descobrirem fraude no processo.


 Municípios: ['Chapecó']
 Modalidades: ['Pregão Eletrônico']


In [37]:
# Exemplo 2
noticia_2 = """
em irani, dois empresários denunciados pelo mpsc são condenados por fraude em licitação eles participaram do certame apenas para simular uma disputa, garantindo que cada um ficasse com parte do contrato sem competição efetiva. dois empresários denunciados pelo ministério público de santa catarina (mpsc) foram condenados por fraudar uma licitação em irani, no oeste do estado. dario francisco bresola e vanderlei biagentini foram condenados à pena de dois anos de detenção, convertida em duas penas restritivas de direitos (prestação de serviços comunitários pelo período da condenação e pagamento de um salário mínimo). de acordo com o processo, a investigação conduzida pelo grupo de atuação especial de combate às organizações criminosas (gaeco) revelou que, entre maio e junho de 2018, os empresários manipularam o pregão presencial n. 36/2018, que tinha como objeto o registro de preços para a contratação de serviços de horas-máquina, como motoniveladoras e rolos compactadores. a fraude ocorreu por meio de um ajuste prévio entre as empresas dos acusados, que combinaram dividir os lotes da licitação. vanderlei propôs a dario que cada um ficasse com determinados itens, de modo que um desistisse para garantir a vitória do outro. dessa forma, não houve uma concorrência real, prejudicando a obtenção de preços mais vantajosos para o município. na prática, isso significou que os acusados participaram da licitação apenas para simular uma disputa, garantindo que cada um ficasse com parte do contrato sem competição efetiva. o esquema resultou na adjudicação fraudulenta de contratos para suas respectivas empresas. cabe recurso da sentença. autos n. 5002615-69.2023.8.24.0019
"""

print("\n\nNOTÍCIA 2:")
print(noticia_2)
print("\n" + "="*60)

municipios = encontrar_municipios(noticia_2, solr_municipios)
modalidades = encontrar_modalidades(noticia_2, solr_modalidades)

print("\n Municípios:", municipios)
print(" Modalidades:", modalidades)



NOTÍCIA 2:

em irani, dois empresários denunciados pelo mpsc são condenados por fraude em licitação eles participaram do certame apenas para simular uma disputa, garantindo que cada um ficasse com parte do contrato sem competição efetiva. dois empresários denunciados pelo ministério público de santa catarina (mpsc) foram condenados por fraudar uma licitação em irani, no oeste do estado. dario francisco bresola e vanderlei biagentini foram condenados à pena de dois anos de detenção, convertida em duas penas restritivas de direitos (prestação de serviços comunitários pelo período da condenação e pagamento de um salário mínimo). de acordo com o processo, a investigação conduzida pelo grupo de atuação especial de combate às organizações criminosas (gaeco) revelou que, entre maio e junho de 2018, os empresários manipularam o pregão presencial n. 36/2018, que tinha como objeto o registro de preços para a contratação de serviços de horas-máquina, como motoniveladoras e rolos compactadores. 

In [50]:
caminho_csv = "extractionv0.csv"  

df_noticias = pd.read_csv(caminho_csv)
df_noticias=df_noticias.drop(columns=[ 'link_licitação_painel',
       'data_publicacao_edital', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16',
       'modalidade_licitacao_presente', 'objeto_presente.1'])
print(f"CSV carregado com sucesso!")
print(f"Total de notícias: {len(df_noticias)}")
print(f"\nColunas disponíveis:")
for col in df_noticias.columns:
    print(f"      - {col}")
print(f"\nPrimeiras 3 linhas:")
print(df_noticias.head(3))

CSV carregado com sucesso!
Total de notícias: 983

Colunas disponíveis:
      - link_noticia
      - titulo
      - texto_noticia
      - municipio_ente
      - edital
      - edital_presente
      - modalidade_licitacao
      - modalidade_presente
      - objeto
      - objeto_presente
      - unidade_gestora
      - unidade_gestora_presente

Primeiras 3 linhas:
                                        link_noticia  \
0  https://www.mpsc.mp.br/noticias/em-irani-dois-...   
1  https://www.mpsc.mp.br/noticias/com-a-morte-de...   
2  https://www.mpsc.mp.br/noticias/operacao-caron...   

                                              titulo  \
0  em irani, dois empresários denunciados pelo mp...   
1  com a morte de pai investigado por superfatura...   
2  operação caronte: após pedido do mpsc, municíp...   

                                       texto_noticia municipio_ente   edital  \
0  em irani, dois empresários denunciados pelo mp...          Irani  36/2018   
1  com a morte de pai in

In [58]:
# Reconectar aos cores
solr_municipios = pysolr.Solr(f'{SOLR_URL_BASE}/municipios', timeout=10)
solr_modalidades = pysolr.Solr(f'{SOLR_URL_BASE}/modalidades', timeout=10)

coluna_texto='texto_noticia'

# Criar colunas para armazenar os resultados
df_noticias['municipios_extraidos'] = None
df_noticias['modalidades_extraidas'] = None
df_noticias['editais_extraidos'] = None 

print(f"Processando {len(df_noticias)} notícias...\n")

for idx, row in df_noticias.iterrows():
    texto = str(row[coluna_texto])
    
    # Extrair municípios, modalidades e editais
    municipios = encontrar_municipios(texto, solr_municipios)
    modalidades = encontrar_modalidades(texto, solr_modalidades)
    editais = encontrar_editais(texto)  # NOVA EXTRAÇÃO
    
    # Armazenar diretamente
    df_noticias.at[idx, 'municipios_extraidos'] = municipios if municipios else []
    df_noticias.at[idx, 'modalidades_extraidas'] = modalidades if modalidades else []
    df_noticias.at[idx, 'editais_extraidos'] = editais if editais else []  # NOVA LINHA
    
    # Exibir progresso a cada 50 notícias
    if (idx + 1) % 50 == 0:
        print(f" {idx + 1} notícias processadas...")

print(f"\nProcessamento concluído!")
print(f"\n   Exemplo de resultado (primeira notícia com extração):")
primeira_com_dados = df_noticias[
    (df_noticias['municipios_extraidos'].apply(len) > 0) | 
    (df_noticias['modalidades_extraidas'].apply(len) > 0)
].iloc[0]

print(f"\n   Texto: {primeira_com_dados[coluna_texto][:100]}...")
print(f"\n   Municípios: {primeira_com_dados['municipios_extraidos']}")
print(f"   Modalidades: {primeira_com_dados['modalidades_extraidas']}")

Processando 983 notícias...

 50 notícias processadas...
 100 notícias processadas...
 150 notícias processadas...
 200 notícias processadas...
 250 notícias processadas...
 300 notícias processadas...
 350 notícias processadas...
 400 notícias processadas...
 450 notícias processadas...
 500 notícias processadas...
 550 notícias processadas...
 600 notícias processadas...
 650 notícias processadas...
 700 notícias processadas...
 750 notícias processadas...
 800 notícias processadas...
 850 notícias processadas...
 900 notícias processadas...
 950 notícias processadas...

Processamento concluído!

   Exemplo de resultado (primeira notícia com extração):

   Texto: em irani, dois empresários denunciados pelo mpsc são condenados por fraude em licitação eles partici...

   Municípios: ['Irani']
   Modalidades: ['Concorrência', 'Pregão Presencial']


In [ ]:
# Salvar o CSV com os resultados da extração utilizando Apache Solr
arquivo_saida = "extractionv0_Solr_3_atributos.csv"

colunas_int= [
'unidade_gestora_presente',
    'objeto_presente',
    'modalidade_presente',
    'edital_presente']

df_noticias[colunas_int] = df_noticias[colunas_int].astype('Int64')

df_noticias.to_csv(arquivo_saida, index=False)
print(f"Resultados salvos em: {arquivo_saida}")

# Análise básica
print("ANÁLISE DOS RESULTADOS")

total_noticias = len(df_noticias)
noticias_com_municipio = df_noticias['municipios_extraidos'].apply(len).gt(0).sum()
noticias_com_modalidade = df_noticias['modalidades_extraidas'].apply(len).gt(0).sum()
noticias_com_edital = df_noticias['editais_extraidos'].apply(len).gt(0).sum()

print(f"\nTotal de notícias: {total_noticias}")
print(f"Notícias com município extraído: {noticias_com_municipio} ({noticias_com_municipio/total_noticias*100:.1f}%)")
print(f"Notícias com modalidade extraída: {noticias_com_modalidade} ({noticias_com_modalidade/total_noticias*100:.1f}%)")
print(f"Notícias com edital extraído: {noticias_com_edital} ({noticias_com_edital/total_noticias*100:.1f}%)")


Resultados salvos em: extractionv0_extração_v0.csv
ANÁLISE DOS RESULTADOS

Total de notícias: 983
Notícias com município extraído: 850 (86.5%)
Notícias com modalidade extraída: 286 (29.1%)
Notícias com edital extraído: 151 (15.4%)
